In [1]:
# import pandas as pd
# df = pd.read_csv('IoT_Network_Intrusion_Dataset.csv', encoding='utf-8')
# df.info()

In [2]:
# df.dtypes

In [3]:
import pandas as pd
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from scipy.stats import mode
import time
import os
from joblib import dump, load
import os
from scipy.stats import mode
from tqdm import tqdm
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
    

SEED = 42
NUM_FEATURES = 13

In [4]:
# model = load('quantum_meta_model.joblib')

# qsvm_model = model["qsvm"]
# qnn_model = model["qnn"]
# meta_model = model["meta_model"]
# label_encoders = model["label_encoders"]
# scaler = model["scaler"]
# feature_columns = model["feature_columns"]


In [5]:
# categorical_cols = df.select_dtypes(include='object').columns
# label_encoders = {}

# # X = df.drop('Cat', axis=1)
# # X = df.drop('Sub_Cat', axis=1)

# le = LabelEncoder()
# for col in categorical_cols:
#     df[col] = le.fit_transform(df[col].astype(str))
#     label_encoders[col] = le
#     print(f"\nColumn: {col}")
#     print(dict(zip(le.classes_, le.transform(le.classes_))))

# feature_names = df.columns

# X = df.drop('Label', axis=1)
# X.drop('Cat', axis=1)
# X.drop('Sub_Cat', axis=1)
# y = df['Label']


In [6]:



# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
# print(f"Original class distribution: {np.bincount(y_train)}")

# import numpy as np

# X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
# X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)


# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)


# pca = PCA(n_components=NUM_FEATURES, random_state=SEED)
# X_train_pca = pca.fit_transform(X_train)
# X_test_pca = pca.transform(X_test_scaled)

# print("Features Selected: ")
# for i, component in enumerate(pca.components_):
#     # Get the index of the feature with the largest absolute value in this component
#     important_idx = np.argmax(np.abs(component))
#     print(f"Principal Component {i+1}: most important feature is '{feature_names[important_idx]}'")


# X_train = X_train_pca
# X_test = X_test_pca

# print(X_train.shape)
# print(X_test.shape)


In [7]:
# import pandas as pd
# import numpy as np

# def preprocess_new_data(df, label_encoders, scaler, pca, feature_columns):
#     """
#     Preprocess a new dataset using the same logic as training.

#     Parameters:
#     - df: DataFrame of new data
#     - label_encoders: dict of LabelEncoder objects from training
#     - scaler: StandardScaler object from training
#     - pca: PCA object from training
#     - feature_columns: list of feature names used in training

#     Returns:
#     - X_pca: PCA-transformed feature matrix ready for prediction
#     """

#     # Ensure same columns as training (order & missing)
#     df = df.reindex(columns=feature_columns)

#     # Apply label encoding for categorical columns
#     for col, le in label_encoders.items():
#         if col in df.columns:
#             df[col] = le.transform(df[col].astype(str))
#         else:
#             df[col] = 0  # handle missing categorical col by filling with 0

#     # Apply scaling
#     df_scaled = scaler.transform(df)

#     # Apply PCA
#     X_pca = pca.transform(df_scaled)

#     return X_pca


In [8]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report

# --- Load saved model bundle ---
model_bundle = joblib.load("quantum_meta_model.joblib")

qsvm_model = model_bundle["qsvm"]
qnn_model = model_bundle["qnn"]
meta_model = model_bundle["meta_model"]
label_encoders = model_bundle["label_encoders"]
scaler = model_bundle["scaler"]
pca = model_bundle["pca"]  # Make sure you saved PCA in your training notebook
feature_columns = model_bundle["feature_columns"]

# --- Preprocessing function ---
def preprocess_new_data(df, label_encoders, scaler, pca, feature_columns):
    # Ensure same columns as training
    df = df.reindex(columns=feature_columns)

    # Encode categorical variables
    for col, le in label_encoders.items():
        if col in df.columns:
            # Fill NaN with a placeholder string
            df[col] = df[col].fillna("unknown").astype(str)

            # Handle unseen categories
            df[col] = df[col].apply(lambda x: x if x in le.classes_ else "unknown")

            # If "unknown" not in classes_, we must add it to LabelEncoder
            if "unknown" not in le.classes_:
                import numpy as np
                le.classes_ = np.append(le.classes_, "unknown")

            df[col] = le.transform(df[col])
        else:
            # If column missing entirely, fill with 0
            df[col] = 0

    # Scale numeric features
    df_scaled = scaler.transform(df)

    # Apply PCA
    X_pca = pca.transform(df_scaled)

    return X_pca


# --- Load new dataset ---
new_data = pd.read_csv("IoT_Network_Intrusion_Dataset.csv")

# If you have labels in your new dataset
y_true = None
if "Label" in new_data.columns:
    y_true = new_data["Label"]
    new_data = new_data.drop(columns=["Label"])
    new_data = new_data.drop(columns=["Cat"])
    new_data = new_data.drop(columns=["Sub_Cat"])

if "label" in new_data.columns:
    y_true = new_data["label"]
    new_data = new_data.drop(columns=["label"])
    new_data = new_data.drop(columns=["Cat"])
    new_data = new_data.drop(columns=["Sub_Cat"])
    

# --- Preprocess ---
X_new = preprocess_new_data(new_data, label_encoders, scaler, pca, feature_columns)

print("Preprocessing Complete")


# --- Predictions ---
qsvm_preds = qsvm_model.predict(X_new[3000])
print("QSVM Complete")
qnn_preds = qnn_model.predict(X_new[3000])
print("QNN Complete")

meta_input = np.column_stack((qsvm_preds, qnn_preds))
final_preds = meta_model.predict(meta_input)

# --- Output ---
print("Predictions:", final_preds)

# --- Classification Report if labels available ---
if y_true is not None:
    print("\nClassification Report:")
    print(classification_report(y_true, final_preds))


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- label
